In [ ]:
import os
import sys
import json
import torch
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams
from gpu_memory import GPUMonitor

# Add project modules
sys.path.append(os.path.abspath("schema"))
from helper import load_data, load_shots, get_prompt, ORDER_MAIN, SUBJECT_FIELDS, TREATMENT_FIELDS, save_llm_results
from schema import get_schema_main, get_schema_sub
from splitter import find_valid_phrases_list

# Config
SPLITS = [1, 2, 3, 4, 5]
N_SHOTS = 50
MODEL_NAME = "google/medgemma-27b-it" 

# Initialize LLM once
llm = LLM(
    model=MODEL_NAME,
    dtype=torch.bfloat16,
    trust_remote_code=True,
    max_model_len=int(8192*1.5),
    seed=0
)

gpu_monitor = GPUMonitor()

/home/hellwig/miniconda3/envs/vllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
for split in SPLITS:
    print(f"\n{'='*20} PROCESSING SPLIT {split} {'='*20}")
    
    DATA_DIR = f"/home/hellwig/medical/phee-with-chatgpt/data/converted_data/text2spotasoc/event/phee2_cross{split}"
    TEST_PATH = os.path.join(DATA_DIR, "test.json")

    # 1. Load data
    test_data = load_data(TEST_PATH)
    print(f"Loaded {len(test_data)} test examples for Split {split}.")

    # 2. Get fixed few-shot examples
    raw_shots = load_shots(n_shot=N_SHOTS, cross_split=split, format="raw")
    formatted_main_shots = load_shots(n_shot=N_SHOTS, cross_split=split, format="main")
    formatted_sub_shots = load_shots(n_shot=N_SHOTS, cross_split=split, format="sub")

    shots_main = []
    shots_sub = []
    for i in range(N_SHOTS):
        text = raw_shots[i]["text"]
        shots_main.append((text, formatted_main_shots[i]))
        shots_sub.append((text, formatted_sub_shots[i]))

    # 3. Generate Prompts & Schema Patterns
    prompts_main = []
    prompts_sub = []
    schema_patterns_main = []
    schema_patterns_sub = []

    for entry in test_data:
        sentence = entry["text"]
        
        # MAIN
        prompts_main.append(get_prompt("main", shots_main, sentence, ORDER_MAIN))
        
        # SUB
        prompts_sub.append(get_prompt("sub", shots_sub, sentence, ORDER_MAIN, SUBJECT_FIELDS, TREATMENT_FIELDS))
        
        # Regex
        valid_phrases = find_valid_phrases_list(sentence, 64)
        schema_patterns_main.append(get_schema_main(valid_phrases))
        schema_patterns_sub.append(get_schema_sub(valid_phrases))

    # 4. Generate for MAIN task
    main_params = [
        SamplingParams(
            temperature=0.0,
            max_tokens=512,
            structured_outputs=StructuredOutputsParams(regex=pattern),
            logprobs=10,
            seed=0,
            stop=")]"
        ) for pattern in schema_patterns_main
    ]

    print(f"Starting MAIN task for Split {split}...")
    gpu_monitor.start()
    results_main = llm.generate(prompts_main, sampling_params=main_params)
    watt_main, dur_main = gpu_monitor.stop()
    
    preds_main = []
    for res in results_main:
        t = res.outputs[0].text
        if not t.endswith(")]"): t += ")]"
        preds_main.append(t)
    
    save_llm_results(preds_main, results_main, {"mean_watt": watt_main, "duration": dur_main}, 
                     filename=f"split{split}_main", task_type="main")

    # 5. Generate for SUB task
    sub_params = [
        SamplingParams(
            temperature=0.0,
            max_tokens=1024,
            structured_outputs=StructuredOutputsParams(regex=pattern),
            logprobs=10,
            seed=0,
            stop="}]"
        ) for pattern in schema_patterns_sub
    ]

    print(f"Starting SUB task for Split {split}...")
    gpu_monitor.start()
    results_sub = llm.generate(prompts_sub, sampling_params=sub_params)
    watt_sub, dur_sub = gpu_monitor.stop()
    
    preds_sub = []
    for res in results_sub:
        t = res.outputs[0].text
        if not t.endswith("}]"): t += "}]"
        preds_sub.append(t)
    
    save_llm_results(preds_sub, results_sub, {"mean_watt": watt_sub, "duration": dur_sub}, 
                     filename=f"split{split}_sub", task_type="sub")

print("\nALL SPLITS COMPLETED.")


==================== PROCESSING SPLIT 1 ====================
Loaded 968 test examples for Split 1.
Starting MAIN task for Split 1...
GPU-Monitoring gestartet...


Processed prompts: 100%|██████████| 968/968 [00:53<00:00, 17.95it/s, est. speed input: 58602.79 toks/s, output: 525.60 toks/s]


GPU-Monitoring gestoppt.
Messdauer: 66.26 Sekunden
Anzahl Messungen: 517
Durchschnittlicher Verbrauch: 583.89 W
Results saved to results/split1_main.txt and results/split1_main.json
Starting SUB task for Split 1...
GPU-Monitoring gestartet...


Processed prompts: 100%|██████████| 968/968 [02:14<00:00,  7.19it/s, est. speed input: 48158.58 toks/s, output: 690.24 toks/s] 


GPU-Monitoring gestoppt.
Messdauer: 204.20 Sekunden
Anzahl Messungen: 1566
Durchschnittlicher Verbrauch: 596.00 W
Results saved to results/split1_sub.txt and results/split1_sub.json

==================== PROCESSING SPLIT 2 ====================
Loaded 967 test examples for Split 2.
Starting MAIN task for Split 2...
GPU-Monitoring gestartet...


Processed prompts: 100%|██████████| 967/967 [00:50<00:00, 19.10it/s, est. speed input: 62763.93 toks/s, output: 508.58 toks/s]


GPU-Monitoring gestoppt.
Messdauer: 63.17 Sekunden
Anzahl Messungen: 495
Durchschnittlicher Verbrauch: 593.55 W
Results saved to results/split2_main.txt and results/split2_main.json
Starting SUB task for Split 2...
GPU-Monitoring gestartet...


Processed prompts:  29%|██▉       | 281/967 [00:35<01:32,  7.39it/s, est. speed input: 53065.72 toks/s, output: 640.28 toks/s] 

## LLM Initialization
Loading the model using vLLM.

## Execution with Constrained Decoding
Running the inference for both 'main' and 'sub' tasks using the generated regex patterns.

In [ ]:
predictions_sub[0]